# Camada prata: limpeza e padronização

Este notebook cria as tabelas da camada prata (`pospucrio.silver`) a partir da camada bronze. Ele faz parte do pipeline: é o segundo passo, depois da ingestão (bronze) e antes da modelagem (ouro).

**O que ele faz:** para cada tabela usada no projeto, lê o dado bruto, aplica tipagem correta, padronização e as correções decididas a partir do diagnóstico de qualidade, e grava o resultado como tabela Delta.

**O que ele não faz:** investigação. As consultas que encontraram os problemas e justificam cada regra estão no notebook `Analises`. Aqui ficam apenas as transformações, cada uma com a explicação do que faz e por quê.

**Como rodar:** sempre do início ao fim. As tabelas de clientes e vendedores dependem de uma função e de uma view temporária criadas nas primeiras células; a view existe apenas durante a sessão do notebook.

**Reexecução:** todas as células usam `CREATE OR REPLACE`. Rodar o notebook várias vezes produz sempre o mesmo resultado, sem duplicar dados.

**Convenção de tipagem:** aqui se usa `CAST`, e não `try_cast`. O diagnóstico confirmou que todos os valores convertem; se uma carga futura trouxer um valor inválido, o pipeline falha e avisa, em vez de gravar nulo silenciosamente.

## 0. Preparação

Objetos compartilhados entre as tabelas: o esquema de destino, a função de padronização de cidades e a lista de correções pontuais.

### 0.1 Esquema da camada prata

Cria o esquema onde as tabelas limpas são gravadas, caso ainda não exista.

In [0]:
CREATE SCHEMA IF NOT EXISTS pospucrio.silver;

### 0.2 Função de padronização de cidades

Aplicada às colunas de cidade de clientes e vendedores. Centralizar a regra numa função garante que a mesma cidade seja escrita da mesma forma nas duas tabelas; uma limpeza diferente em cada uma impediria qualquer cruzamento por cidade.

A função fica gravada no catálogo (`pospucrio.silver.padronizar_cidade`) e é aplicada em etapas, lidas de dentro para fora:

1. Converte para minúsculas e remove espaços nas bordas.
2. Remove marcas de acentuação soltas (`\p{M}`). Alguns textos gravam "ã" como "a" seguido de um til separado; visualmente é igual, mas não seria alcançado pela etapa seguinte.
3. Troca letras acentuadas pelas equivalentes sem acento, e o acento agudo ou grave usado como apóstrofo ("d´oeste") pelo apóstrofo.
4. Remove complementos entre parênteses ("arraial d'ajuda (porto seguro)").
5. Corta tudo a partir do primeiro separador (barra, barra invertida, vírgula ou traço cercado de espaços), eliminando o estado digitado junto com a cidade ("jacarei / sao paulo", "rio de janeiro, rio de janeiro, brasil"). O traço só conta com espaço dos dois lados, para preservar nomes oficiais com hífen, como "embu-guacu".
6. Reduz espaços múltiplos a um.
7. Restaura o apóstrofo em nomes que o trazem na origem ora com apóstrofo, ora com espaço ("dias d avila" vira "dias d'avila").

In [0]:
CREATE OR REPLACE FUNCTION pospucrio.silver.padronizar_cidade(c STRING)
RETURNS STRING
RETURN regexp_replace(
         trim(regexp_replace(
           regexp_replace(
             regexp_replace(
               translate(
                 regexp_replace(lower(trim(c)), '\\p{M}', ''),
                 "áàâãäéèêëíìîïóòôõöúùûüç´`",
                 "aaaaaeeeeiiiiooooouuuuc''"),
               '\\s*\\(.*\\)', ''),
             '\\s*(/|\\\\|,| - ).*$', ''),
           '\\s+', ' ')),
         '\\bd (?=[aeiou])', "d'");

### 0.3 Correções pontuais de cidade

Casos que nenhuma regra geral resolve sem estragar outros nomes. Ficam numa view temporária, que existe só durante a execução do notebook, em vez de numa tabela gravada: a lista é usada apenas aqui, e não há motivo para mantê-la no catálogo.

| Estado | Valor após a função | Correção | Motivo |
|---|---|---|---|
| SP | mogi-mirim | mogi mirim | Hífen não é oficial neste município. Não pode virar regra, porque em outros (Embu-Guaçu, Guajará-Mirim) ele é oficial. |
| SP | portoferreira | porto ferreira | Espaço faltando. |
| SP | sao paulo sp | sao paulo | Estado digitado sem separador. |
| SP | sbc | sao bernardo do campo | Sigla, que sobra depois de a função cortar "/sp". |
| RJ | 04482255 | nulo | Número no lugar da cidade. |
| PR | vendas@creditparts.com.br | nulo | E-mail no lugar da cidade. |
| SP | sp | nulo | Apenas a sigla do estado, sem cidade. |

As três últimas são anuladas porque o texto não informa a cidade. A consulta pelo prefixo de CEP do vendedor na tabela de geolocalização indicaria rio de janeiro, sao paulo e maringa, respectivamente (ver `Analises`, seção 2.6), mas optou-se por não inferir o valor a partir de outra fonte, já que a cidade não é usada no modelo da camada ouro.

In [0]:
CREATE OR REPLACE TEMP VIEW correcao_cidade AS
SELECT * FROM VALUES
  ('SP', 'mogi-mirim',                'mogi mirim'),
  ('SP', 'portoferreira',             'porto ferreira'),
  ('SP', 'sao paulo sp',              'sao paulo'),
  ('SP', 'sbc',                       'sao bernardo do campo'),
  ('RJ', '04482255',                  CAST(NULL AS STRING)),  
  ('PR', 'vendas@creditparts.com.br', CAST(NULL AS STRING)),
  ('SP', 'sp',                        CAST(NULL AS STRING))
AS t(uf, cidade_origem, cidade_correta);

## 1. customers

**Granularidade:** um `customer_id` por linha (99.441). No Olist, o `customer_id` é gerado a cada pedido; a pessoa é identificada pelo `customer_unique_id` (96.096 pessoas distintas).

**Transformações:**
- Espaços removidos das bordas de IDs e textos.
- CEP completado com zeros à esquerda até 5 dígitos, garantindo que prefixos iniciados por zero não percam o dígito.
- UF em maiúsculas.
- Cidade padronizada pela função da seção 0.2 e, em seguida, pelas correções da seção 0.3.

**Sobre o join com as correções:** o `CASE` verifica se a cidade foi encontrada na lista (`r.uf IS NOT NULL`). Se foi, vale o valor da lista, inclusive quando ele é nulo; se não foi, fica a cidade padronizada. Um `coalesce` não serviria, porque trataria uma correção nula como "não encontrada" e manteria o valor errado.

**Diagnóstico que sustenta:** UFs todas válidas; 6 cidades com duas grafias, das quais 5 por apóstrofo e 1 por hífen; IDs no formato esperado; nenhum pedido sem cliente.

In [0]:
CREATE OR REPLACE TABLE pospucrio.silver.customers AS
WITH base AS (
  SELECT
    trim(customer_id)                                  AS customer_id,
    trim(customer_unique_id)                           AS customer_unique_id,
    lpad(trim(customer_zip_code_prefix), 5, '0')       AS customer_zip_code_prefix,  -- CEP sempre com 5 dígitos
    pospucrio.silver.padronizar_cidade(customer_city)  AS customer_city,             -- regra geral
    upper(trim(customer_state))                        AS customer_state
  FROM pospucrio.bronze.olist_customers_dataset
)
SELECT
  b.customer_id,
  b.customer_unique_id,
  b.customer_zip_code_prefix,
  CASE WHEN r.uf IS NOT NULL THEN r.cidade_correta    -- encontrada na lista de correções
       ELSE b.customer_city END AS customer_city,     -- demais: cidade padronizada
  b.customer_state
FROM base b
LEFT JOIN correcao_cidade r
  ON b.customer_state = r.uf AND b.customer_city = r.cidade_origem;

## 2. sellers

**Granularidade:** um `seller_id` por linha (3.095), todos com ao menos uma venda.

**Transformações:** as mesmas de clientes. A cidade dos vendedores é digitada no cadastro e chega mais suja: estado junto com a cidade, sigla, CEP e e-mail no campo. A maior parte é resolvida pela função (seção 0.2); os demais casos, pela lista de correções (seção 0.3).

**Diagnóstico que sustenta:** 23 UFs, todas válidas; 23 cidades com caracteres fora do padrão e 5 com grafias divergentes antes da padronização.

In [0]:
CREATE OR REPLACE TABLE pospucrio.silver.sellers AS
WITH base AS (
  SELECT
    trim(seller_id)                                  AS seller_id,
    lpad(trim(seller_zip_code_prefix), 5, '0')       AS seller_zip_code_prefix,
    pospucrio.silver.padronizar_cidade(seller_city)  AS seller_city,
    upper(trim(seller_state))                        AS seller_state
  FROM pospucrio.bronze.olist_sellers_dataset
)
SELECT
  b.seller_id,
  b.seller_zip_code_prefix,
  CASE WHEN r.uf IS NOT NULL THEN r.cidade_correta
       ELSE b.seller_city END AS seller_city,
  b.seller_state
FROM base b
LEFT JOIN correcao_cidade r
  ON b.seller_state = r.uf AND b.seller_city = r.cidade_origem;

## 3. orders

**Granularidade:** um pedido por linha (99.441), com um `customer_id` por pedido.

**Transformações:** todas as datas convertidas para timestamp. Nenhum valor alterado ou removido.

**Inconsistências mantidas e documentadas**, por não afetarem as perguntas do objetivo:
- 8 pedidos entregues sem data de entrega e 6 cancelados com data de entrega.
- 1.359 postagens registradas antes da aprovação do pagamento, provável defasagem entre sistemas.
- 23 entregas registradas antes da postagem, erro de registro.

**Pedidos em status intermediário** (shipped, processing, invoiced, approved, created) aparecem em volume estável desde janeiro de 2017, e não apenas no fim do período. São pedidos que pararam e nunca foram concluídos. Por isso, na camada ouro, apenas pedidos com status `delivered` contam como compra efetivada.

In [0]:
CREATE OR REPLACE TABLE pospucrio.silver.orders AS
SELECT
  trim(order_id)                                        AS order_id,
  trim(customer_id)                                     AS customer_id,
  trim(order_status)                                    AS order_status,
  CAST(order_purchase_timestamp AS TIMESTAMP)           AS order_purchase_timestamp,
  CAST(order_approved_at AS TIMESTAMP)                  AS order_approved_at,
  CAST(order_delivered_carrier_date AS TIMESTAMP)       AS order_delivered_carrier_date,
  CAST(order_delivered_customer_date AS TIMESTAMP)      AS order_delivered_customer_date,
  CAST(order_estimated_delivery_date AS TIMESTAMP)      AS order_estimated_delivery_date
FROM pospucrio.bronze.olist_orders_dataset;

## 4. order_items

**Granularidade:** um item por pedido, chave composta `order_id` + `order_item_id` (112.650 linhas, sem duplicatas). Cada linha é uma unidade vendida.

**Transformações:** número do item como inteiro, prazo de postagem como timestamp, preço e frete como decimal com duas casas. Nenhum valor alterado.

**Mantidos e documentados:**
- 383 itens com frete zero, plausível como frete grátis.
- 4 prazos de postagem em 2020, cerca de 1.050 dias após a compra, contra mediana de 6 dias. Erro de registro; mantidos porque a coluna não é usada no modelo.
- Outliers de preço (até R$ 6.735) e de frete (até R$ 409,68), verificados como coerentes com categoria, peso e volume dos produtos.

In [0]:
CREATE OR REPLACE TABLE pospucrio.silver.order_items AS
SELECT
  trim(order_id)                            AS order_id,
  CAST(order_item_id AS INT)                AS order_item_id,
  trim(product_id)                          AS product_id,
  trim(seller_id)                           AS seller_id,
  CAST(shipping_limit_date AS TIMESTAMP)    AS shipping_limit_date,
  CAST(price AS DECIMAL(10,2))              AS price,          -- dinheiro sempre em decimal, nunca em float
  CAST(freight_value AS DECIMAL(10,2))      AS freight_value
FROM pospucrio.bronze.olist_order_items_dataset;

## 5. order_payments

**Granularidade:** um pagamento por linha, chave composta `order_id` + `payment_sequential` (103.886 pagamentos de 99.440 pedidos). Um pedido pode ter vários pagamentos, de meios iguais ou diferentes.

**Transformações:** sequência e parcelas como inteiro, valor como decimal.

**Parcelas zeradas convertidas em nulo:** 2 pagamentos com cartão de crédito em pedidos entregues. Todo pagamento tem ao menos uma parcela, então zero é valor impossível; como os pedidos foram entregues, trata-se de erro de registro, não de cancelamento. Manter o zero deixaria um valor inválido numa camada que deve ser confiável.

**Mantidos sem alteração:**
- 3 pagamentos `not_defined` de valor zero, todos de pedidos cancelados.
- 6 vouchers de valor zero em pedidos entregues ou enviados. Não somam ao valor pago e o voucher não entra na classificação de parcelamento, então não afetam nenhuma regra.

In [0]:
CREATE OR REPLACE TABLE pospucrio.silver.order_payments AS
SELECT
  trim(order_id)                                   AS order_id,
  CAST(payment_sequential AS INT)                  AS payment_sequential,
  trim(payment_type)                               AS payment_type,
  CASE WHEN CAST(payment_installments AS INT) = 0 THEN NULL   -- zero parcelas é inválido
       ELSE CAST(payment_installments AS INT) END  AS payment_installments,
  CAST(payment_value AS DECIMAL(10,2))             AS payment_value
FROM pospucrio.bronze.olist_order_payments_dataset;

## 6. order_reviews

**Granularidade:** chave composta `review_id` + `order_id` (99.224 pares distintos). Nenhuma das duas colunas é única sozinha.

**Transformações:**
- Nota como inteiro.
- Comentários e títulos com espaços removidos; texto vazio convertido em nulo (`nullif`).
- Data de criação convertida para `DATE`. É a data de envio da pesquisa de satisfação, e a origem nunca registra a hora (sempre 00:00:00); guardar como timestamp sugeriria uma precisão que o dado não tem.
- Data de resposta como timestamp, com hora completa.

**Nenhuma linha removida.** As repetições são mantidas nesta camada:
- 789 avaliações associadas a mais de um pedido, sempre do mesmo cliente (em 95% dos casos, da mesma data de compra): uma avaliação única de uma compra dividida em vários pedidos.
- 547 pedidos com mais de uma avaliação.

A escolha de uma avaliação por pedido é feita na camada ouro: prevalece a de resposta mais recente (`review_answer_timestamp`); em empate, a de criação mais recente; persistindo, o maior `review_id`.

In [0]:
CREATE OR REPLACE TABLE pospucrio.silver.order_reviews AS
SELECT
  trim(review_id)                                       AS review_id,
  trim(order_id)                                        AS order_id,
  CAST(review_score AS INT)                             AS review_score,
  nullif(trim(review_comment_title), '')                AS review_comment_title,     -- vazio vira nulo
  nullif(trim(review_comment_message), '')              AS review_comment_message,
  CAST(CAST(review_creation_date AS TIMESTAMP) AS DATE) AS review_creation_date,     -- origem não registra hora
  CAST(review_answer_timestamp AS TIMESTAMP)            AS review_answer_timestamp
FROM pospucrio.bronze.olist_order_reviews_dataset;

## 7. products

**Granularidade:** um produto por linha (32.951, sem duplicatas).

**Transformações:**
- Nomes de coluna corrigidos: `product_name_lenght` e `product_description_lenght`, grafados errado na origem, passam a `product_name_length` e `product_description_length`. Os nomes corrigidos existem da prata em diante; consultas no bronze usam a grafia original.
- Medidas e contagens como inteiro.
- Peso zero (4 produtos) convertido em nulo, por ser valor impossível.

**Categoria mantida nula e com o nome original.** 610 produtos não têm categoria, nome, descrição nem fotos: cadastro incompleto. A prata registra o dado como veio; o rótulo "sem categoria" e a troca de sublinhado por espaço são aplicados na camada ouro, que é de apresentação. Assim a prata preserva o código original da categoria.

**Categorias com sufixo `_2`** (`eletrodomesticos_2`, `casa_conforto_2`) são mantidas separadas das homônimas: os perfis de preço e peso são muito diferentes, indicando categorias distintas e não duplicação.

In [0]:
CREATE OR REPLACE TABLE pospucrio.silver.products AS
SELECT
  trim(product_id)                                  AS product_id,
  nullif(trim(product_category_name), '')           AS product_category_name,
  CAST(product_name_lenght AS INT)                  AS product_name_length,         -- corrige "lenght"
  CAST(product_description_lenght AS INT)           AS product_description_length,  -- corrige "lenght"
  CAST(product_photos_qty AS INT)                   AS product_photos_qty,
  nullif(CAST(product_weight_g AS INT), 0)          AS product_weight_g,            -- peso zero vira nulo
  CAST(product_length_cm AS INT)                    AS product_length_cm,
  CAST(product_height_cm AS INT)                    AS product_height_cm,
  CAST(product_width_cm AS INT)                     AS product_width_cm
FROM pospucrio.bronze.olist_products_dataset;

## 8. Tabelas que permanecem apenas no bronze

**`olist_geolocation_dataset`:** nenhuma pergunta do objetivo usa coordenadas, e a tabela não tem chave (vários registros por prefixo de CEP), o que exigiria uma agregação sem uso no projeto. Continua disponível no bronze como referência; foi usada assim no diagnóstico das cidades de vendedores.

**`product_category_name_translation`:** traduz as categorias para o inglês. Como a análise é feita em português, as categorias são usadas com os nomes originais, o que também dispensa tratar as duas categorias que não têm tradução (`pc_gamer` e `portateis_cozinha_e_preparadores_de_alimentos`).